Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_profiling import ProfileReport
%matplotlib inline

sns.set_theme(style="whitegrid")

In [2]:
DATA_PATH_TRAIN = "../data/train.csv"
DATA_PATH_TEST = "../data/test.csv"
DATA_PATH_CLIENT = "../data/clientes.csv"
TARGET_COLUMN = "repeat_purchase_90d"

df_train = pd.read_csv(DATA_PATH_TRAIN, parse_dates=["purchase_date"])
df_test = pd.read_csv(DATA_PATH_TEST, parse_dates=["purchase_date"])
df_clientes = pd.read_csv(DATA_PATH_CLIENT)

print("train:", df_train.shape, "| test:", df_test.shape, "| clientes:", df_clientes.shape)
print("Columns only in train:", set(df_train.columns) - set(df_test.columns))
print("Duplicate rows:", df_train.duplicated().sum(), "| Duplicate IDs:", df_train["ID"].duplicated().sum())
print("Duplicate customer_id in clientes:", df_clientes["customer_id"].duplicated().sum())

train: (6534, 39) | test: (986, 38) | clientes: (1143, 13)
Columns only in train: {'repeat_purchase_90d'}
Duplicate rows: 0 | Duplicate IDs: 0
Duplicate customer_id in clientes: 0


**Observation:** Each row is one purchase. There are no duplicate rows or IDs, and `clientes` has one row per customer, so a left merge on `customer_id` won't duplicate any rows (this matters in Section 2).

In [3]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6534 entries, 0 to 6533
Data columns (total 39 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Total a Pagar                   6469 non-null   float64       
 1   Total Bruto                     6469 non-null   float64       
 2   Total Liquido                   6469 non-null   float64       
 3   Total Imp_                      6469 non-null   float64       
 4   Total Portes                    6469 non-null   float64       
 5   Total Desc_ Global              6469 non-null   float64       
 6   Total Desc_ Linha               6469 non-null   float64       
 7   Custo                           6469 non-null   float64       
 8   N_ Detalhes                     6469 non-null   float64       
 9   invoice_count_current_purchase  6469 non-null   float64       
 10  repeat_purchase_90d             6534 non-null   int64         
 11  days

In [4]:
df_train["purchase_date"].value_counts()

purchase_date
2015-10-31    52
2018-08-05    46
2016-07-17    42
2015-07-12    41
2018-07-21    40
              ..
2017-06-20     1
2018-06-26     1
2021-05-29     1
2019-01-07     1
2023-01-31     1
Name: count, Length: 1492, dtype: int64

In [5]:
df_train.head(10)

,Total a Pagar,Total Bruto,Total Liquido,Total Imp_,Total Portes,Total Desc_ Global,Total Desc_ Linha,Custo,N_ Detalhes,invoice_count_current_purchase,...,customer_id,payment_type,salesperson,commercial_zone,transport_method,warehouse,document_series,purchase_date,ID,random_noise
0,558.11,510.90,453.75,104.36,0.00,0.0,57.15,153.099911,4.0,1.0,...,660784,Maybe Tomorrow Payment 10,VLOOKUP Master 9,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2024-10-29,10000000,-0.660448
1,978.80,830.02,795.77,183.03,0.00,0.0,34.25,472.943134,32.0,1.0,...,189146,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 36,Final Excel Warehouse 1,Ctrl Z Series 1,2017-07-08,10000001,0.924510
2,10.33,12.00,8.40,1.93,0.00,0.0,3.60,6.097485,1.0,1.0,...,103940,Maybe Tomorrow Payment 9,VLOOKUP Master 1,Decorative Dashboard Zone 4,NaN,Final Excel Warehouse 1,Ctrl Z Series 3,2019-03-19,10000002,-0.258730
3,856.83,696.61,696.61,160.22,0.00,0.0,0.00,408.167275,9.0,1.0,...,512927,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 71,Final Excel Warehouse 1,Ctrl Z Series 3,2018-11-10,10000003,1.448402
4,1616.10,1488.85,1313.90,302.20,0.00,0.0,174.95,485.401864,33.0,1.0,...,122356,Maybe Tomorrow Payment 12,NaN,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 61,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-30,10000004,-1.058890
5,640.78,568.24,520.96,119.82,0.00,0.0,47.28,336.600650,11.0,1.0,...,821421,Maybe Tomorrow Payment 7,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 62,Final Excel Warehouse 1,Ctrl Z Series 2,2023-07-15,10000005,-0.011464
6,122.83,105.55,100.43,22.40,3.05,0.0,5.12,0.000000,5.0,1.0,...,974136,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 53,Final Excel Warehouse 1,Ctrl Z Series 3,2016-12-02,10000007,-0.747859
7,1409.75,1272.44,1146.14,263.61,0.00,0.0,126.30,246.974108,66.0,1.0,...,488020,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 73,Final Excel Warehouse 1,Ctrl Z Series 1,2021-03-27,10000009,-0.716275
8,962.63,786.27,782.63,180.00,0.00,0.0,3.64,496.926279,46.0,1.0,...,796571,Maybe Tomorrow Payment 7,VLOOKUP Master 1,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 40,Final Excel Warehouse 1,Ctrl Z Series 3,2017-11-11,10000011,-0.519772
9,457.95,372.32,372.32,85.63,0.00,0.0,0.00,129.982522,14.0,1.0,...,229943,Maybe Tomorrow Payment 12,VLOOKUP Master 4,Decorative Dashboard Zone 4,Maybe Tomorrow Delivery 77,Final Excel Warehouse 1,Ctrl Z Series 2,2021-11-20,10000012,1.203258


In [8]:
df_train["random_noise"].sort_values()


2741   -3.529210
3543   -3.263267
256    -3.195287
4183   -3.119475
1954   -3.082920
          ...   
6094         NaN
6111         NaN
6249         NaN
6456         NaN
6462         NaN
Name: random_noise, Length: 6534, dtype: float64

In [ ]:
DATA_PATH_CLIENT = "../data/clientes.csv"

df_clientes = pd.read_csv(DATA_PATH_CLIENT)

df_clientes.info()


In [ ]:
df_clientes.head(10)

Missing Values for each of the datasets

In [ ]:
df_train.isna().sum()

In [ ]:
missing_pct = df_train.isna().mean() * 100
missing_pct.sort_values(ascending=False)

In [ ]:
rows_multi_missing = (df_train.isna().sum(axis=1) > 1)

n_rows = rows_multi_missing.sum()
pct_rows = rows_multi_missing.mean() * 100

print(f"Rows with >1 missing value: {n_rows} ({pct_rows:.2f}%)")

In [ ]:
df_clientes.isna().sum()

In [ ]:
missing_pct = df_clientes.isna().mean() * 100
missing_pct.sort_values(ascending=False)

In [ ]:
rows_multi_missing = (df_clientes.isna().sum(axis=1) > 1)

n_rows = rows_multi_missing.sum()
pct_rows = rows_multi_missing.mean() * 100

print(f"Rows with >1 missing value: {n_rows} ({pct_rows:.2f}%)")

In [ ]:
df_clientes["postal_code"].value_counts()

In [ ]:
df_train.describe().T

In [ ]:
df_clientes.describe(include="all").T

In [ ]:
ProfileReport(df_train, title="Training Data Profile", correlations={
            #"auto": {"calculate": True},
            "pearson": {"calculate": True},
            "spearman": {"calculate": True},
            "kendall": {"calculate": True},
            "phi_k": {"calculate": True},
            "cramers": {"calculate": True},
            },).to_notebook_iframe()

In [ ]:
df_merged = df_train.merge(
    df_clientes[["customer_id", "postal_code"]],
    on="customer_id",
    how="left",
)

top_postal = (
    df_merged.groupby("postal_code")["prior_total_spend"]
    .mean()
    .sort_values(ascending=False)
    .head(20)
)

plt.figure(figsize=(10, 7))
top_postal.sort_values().plot(kind="barh")
plt.xlabel("Mean prior total spend")
plt.ylabel("Postal code")
plt.title("Top 20 postal codes by mean prior total spend")
plt.tight_layout()
plt.show()